# SCPA Scraper Notebook

This notebook is self-contained and demonstrates BeautifulSoup scraping logic, sample job output, cleaning, and deduplication. Playwright can be used for JavaScript-heavy sources, but BeautifulSoup is the lightweight request-path choice for the service.

In [1]:
import hashlib
import re
from bs4 import BeautifulSoup

SPACE_RE = re.compile(r"\s+")

def clean_text(value):
    return SPACE_RE.sub(" ", value or "").strip()

def first_text(node, selectors):
    for selector in selectors:
        found = node.select_one(selector)
        text = clean_text(found.get_text(" ", strip=True) if found else "")
        if text:
            return text
    return ""

def content_hash(title, company, location):
    raw = f"{title.lower()}|{company.lower()}|{location.lower()}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

def tag_texts(node):
    tags = []
    for selector in [".tag", ".tags li", "[data-tag]", ".chip", ".badge"]:
        for found in node.select(selector):
            text = clean_text(found.get_text(" ", strip=True) or found.get("data-tag"))
            if text and text.lower() not in {tag.lower() for tag in tags}:
                tags.append(text)
    return tags[:12]

def extract_jobs(html, source_url=None, limit=25):
    soup = BeautifulSoup(html, "html.parser")
    nodes = soup.select("[data-job], .job, .job-card, .job-listing, .vacancy, article, li")
    jobs, seen = [], set()
    duplicate_count = 0
    for node in nodes:
        title = first_text(node, ["[data-title]", ".title", ".job-title", "h1", "h2", "h3", "a"])
        if not title:
            continue
        company = first_text(node, ["[data-company]", ".company", ".employer"])
        location = first_text(node, ["[data-location]", ".location", ".city"])
        description = first_text(node, ["[data-description]", ".description", ".summary", "p"])
        tags = tag_texts(node)
        digest = content_hash(title, company, location)
        if digest in seen:
            duplicate_count += 1
            continue
        seen.add(digest)
        jobs.append({"title": title, "description": description, "company": company, "location": location, "tags": tags, "source_url": source_url, "content_hash": digest})
        if len(jobs) >= limit:
            break
    return {"count": len(jobs), "jobs": jobs, "deduplicated": duplicate_count}


## Sample HTML

In [2]:
sample_html = """
<article class="job-card">
  <h2 class="job-title"> Master of Ceremony </h2>
  <div class="company"> Event Nusantara </div>
  <div class="location"> Jakarta </div>
  <p class="description"> Host seminars and formal public events in English. </p>
  <span class="tag">Public Speaking</span><span class="tag">English</span>
</article>
<article class="job-card">
  <h2 class="job-title">Backend Developer</h2>
  <div class="company">Tekno API</div>
  <div class="location">Remote</div>
  <p class="description">Build FastAPI services and PostgreSQL integrations.</p>
  <span class="tag">Python</span><span class="tag">FastAPI</span>
</article>
<article class="job-card">
  <h2 class="job-title">Backend Developer</h2>
  <div class="company">Tekno API</div>
  <div class="location">Remote</div>
  <p class="description">Duplicate card that should be removed.</p>
  <span class="tag">Python</span>
</article>
"""
result = extract_jobs(sample_html, source_url="sample://local")
result


{'count': 2,
 'jobs': [{'title': 'Master of Ceremony',
   'description': 'Host seminars and formal public events in English.',
   'company': 'Event Nusantara',
   'location': 'Jakarta',
   'tags': ['Public Speaking', 'English'],
   'source_url': 'sample://local',
   'content_hash': '2757a2cedb48f521'},
  {'title': 'Backend Developer',
   'description': 'Build FastAPI services and PostgreSQL integrations.',
   'company': 'Tekno API',
   'location': 'Remote',
   'tags': ['Python', 'FastAPI'],
   'source_url': 'sample://local',
   'content_hash': 'd632f1e0769e0e5e'}],
 'deduplicated': 1}

## Sample Title, Description, Company, Location, Tags Output

In [3]:
for job in result["jobs"]:
    print({key: job[key] for key in ["title", "description", "company", "location", "tags"]})


{'title': 'Master of Ceremony', 'description': 'Host seminars and formal public events in English.', 'company': 'Event Nusantara', 'location': 'Jakarta', 'tags': ['Public Speaking', 'English']}
{'title': 'Backend Developer', 'description': 'Build FastAPI services and PostgreSQL integrations.', 'company': 'Tekno API', 'location': 'Remote', 'tags': ['Python', 'FastAPI']}


## Cleaning and Dedup Assertions

In [4]:
assert result["count"] == 2
assert result["deduplicated"] == 1
assert result["jobs"][0]["title"] == "Master of Ceremony"
assert result["jobs"][0]["company"] == "Event Nusantara"
assert result["jobs"][0]["location"] == "Jakarta"
assert result["jobs"][0]["tags"] == ["Public Speaking", "English"]
print("All scraper notebook assertions passed.")


All scraper notebook assertions passed.
